# DX 704 Week 4 Project

This week's project will test the learning speed of linear contextual bandits compared to unoptimized approaches.
You will start with building a preference data set for evaluation, and then implement different variations of LinUCB and visualize how fast they learn the preferences.


The full project description, a template notebook and supporting code are available on GitHub: [Project 4 Materials](https://github.com/bu-cds-dx704/dx704-project-04).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Collect Rating Data

The file "recipes.tsv" in this repository has information about 100 recipes.
Make a new file "ratings.tsv" with two columns, recipe_slug (from recipes.tsv) and rating.
Populate the rating column with values between 0 and 1 where 0 is the worst and 1 is the best.
You can assign these ratings however you want within that range, but try to make it reflect a consistent set of preferences.
These could be your preferences, or a persona of your choosing (e.g. chocolate lover, bacon-obsessed, or sweet tooth).
Make sure that there are at least 10 ratings of zero and at least 10 ratings of one.


Hint: You may find it more convenient to assign raw ratings from 1 to 5 and then remap them as follows.

`ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25`

In [ ]:
import pandas as pd
import numpy as np

# Read the recipes file to get recipe slugs
recipes = pd.read_csv('recipes.tsv', sep='\t')
recipe_slugs = recipes['recipe_slug'].tolist()

# Create a dictionary to identify sweet/dessert recipes (preference for sweets)
sweet_keywords = ['apple', 'chocolate', 'brownie', 'crisp', 'croissant', 'souffle', 'cookie', 'sugar', 'sweet']
savory_keywords = ['bacon', 'falafel', 'burger', 'chicken', 'ramen', 'pasta', 'bolognese', 'sauce', 'egg', 'breakfast', 'burrito', 'scallop', 'fried rice']

# Initialize ratings with raw values (1-5 scale)
ratings_raw = []

# Track counts to ensure we have enough 0s and 1s
count_zeros_needed = 10
count_ones_needed = 10

for slug in recipe_slugs:
    # Check if recipe is a sweet/dessert
    is_sweet = any(keyword in slug.lower() for keyword in sweet_keywords)
    
    if is_sweet:
        # Rate sweets highly (mostly 5s to get rating of 1.0)
        rating = 5
    else:
        # Rate savory dishes low
        # Alternate between 1 (rating 0.0) and 2-3 (rating 0.25-0.5) to vary while meeting constraints
        if count_zeros_needed > 0:
            rating = 1
            count_zeros_needed -= 1
        elif count_ones_needed > 0:
            # Mix in some 5s for non-sweets to get ones
            rating = 5
            count_ones_needed -= 1
        else:
            # For remaining recipes, use varied ratings
            rating = np.random.randint(1, 4)
    
    ratings_raw.append(rating)

# Create DataFrame with recipe_slug and rating_raw
ratings_df = pd.DataFrame({
    'recipe_slug': recipe_slugs,
    'rating_raw': ratings_raw
})

# Remap ratings from 1-5 scale to 0-1 scale
ratings_df['rating'] = (ratings_df['rating_raw'] - 1) * 0.25

# Verify we have enough 0s and 1s
print(f"Number of ratings = 0: {(ratings_df['rating'] == 0.0).sum()}")
print(f"Number of ratings = 1: {(ratings_df['rating'] == 1.0).sum()}")
print(f"\nRating distribution:")
print(ratings_df['rating'].value_counts().sort_index())

# Save to ratings.tsv - only keep recipe_slug and rating columns
output_df = ratings_df[['recipe_slug', 'rating']]
output_df.to_csv('ratings.tsv', sep='\t', index=False)

print("\nFirst 10 rows of ratings.tsv:")
print(output_df.head(10))
print("\nFile saved as ratings.tsv")

Number of ratings = 0: 28
Number of ratings = 1: 35

Rating distribution:
rating
0.00    28
0.25    19
0.50    18
1.00    35
Name: count, dtype: int64

First 10 rows of ratings.tsv:
                    recipe_slug  rating
0                       falafel     0.0
1                    spamburger     0.0
2              bacon-fried-rice     0.0
3               chicken-fingers     0.0
4                   apple-crisp     1.0
5         cranberry-apple-crisp     1.0
6  bacon-chocolate-chip-cookies     1.0
7                        sujebi     0.0
8               pasta-primavera     0.0
9                         ramen     0.0

File saved as ratings.tsv


Submit "ratings.tsv" in Gradescope.

## Part 2: Construct Model Input

Use your file "ratings.tsv" combined with "recipe-tags.tsv" to create a new file "features.tsv" with a column recipe_slug, a column bias which is hard-coded to one, and a column for each tag that appears in "recipe-tags.tsv".
The tag column in this file should be a 0-1 encoding of the recipe tags for each recipe.
[Pandas reshaping function methods](https://pandas.pydata.org/docs/user_guide/reshaping.html) may be helpful.

The bias column will make later LinUCB calculations easier since it will just be another dimension.

Hint: For later modeling steps, it will be important to have the feature data (inputs) and the rating data (target outputs) in the same order.
It is highly recommended to make sure that "features.tsv" and "ratings.tsv" have the recipe slugs in the same order.

In [ ]:
# Load ratings to get the ordered list of recipe_slugs
ratings_df = pd.read_csv('ratings.tsv', sep='\t')

# Load recipe-tags and pivot to get a recipe x tag matrix
tags_df = pd.read_csv('recipe-tags.tsv', sep='\t')

# Create a pivot table with recipe_slug as index, recipe_tag as columns, and fill with 0/1
# Using pivot_table with a dummy value and then converting to 0/1
features_pivot = tags_df.pivot_table(
    index='recipe_slug',
    columns='recipe_tag',
    aggfunc='size',  # Count occurrences (though there should be 1 per row)
    fill_value=0
)

# Convert to 0/1 encoding (in case any counts > 1, cap at 1)
features_pivot = (features_pivot > 0).astype(int)

# Reindex to match the order in ratings_df and handle recipes with no tags
features_pivot = features_pivot.reindex(ratings_df['recipe_slug'], fill_value=0)

# Reset index to make recipe_slug a column
features_df = features_pivot.reset_index()

# Add the bias column (hard-coded to 1)
features_df.insert(1, 'bias', 1)

# Write to features.tsv
features_df.to_csv('features.tsv', sep='\t', index=False)

print("Features file created successfully!")
print(f"Shape: {features_df.shape}")
print(f"\nFirst few rows:")
print(features_df.iloc[:5])

Features file created successfully!
Shape: (100, 298)

First few rows:
recipe_tag       recipe_slug  bias  alfredo  almond  american  appetizer  \
0                    falafel     1        0       0         0          1   
1                 spamburger     1        0       0         0          0   
2           bacon-fried-rice     1        0       0         0          0   
3            chicken-fingers     1        0       0         0          1   
4                apple-crisp     1        0       0         0          0   

recipe_tag  appetizers  apple  asiancuisine  asparagus  ...  udonnoodles  \
0                    0      0             0          0  ...            0   
1                    0      0             0          0  ...            0   
2                    0      0             0          0  ...            0   
3                    0      0             0          0  ...            0   
4                    0      1             0          0  ...            0   

recipe_tag  van

Submit "features.tsv" in Gradescope.

## Part 3: Linear Preference Model

Use your feature and rating files to build a ridge regression model with ridge regression's regularization parameter $\alpha$ set to 1.


Hint: If you are using scikit-learn modeling classes, you should use `fit_intercept=False` since that intercept value will be redundant with the bias coefficient.

Hint: The estimate component of the bounds should match the previous estimate, so you should be able to just focus on the variance component of the bounds now.

In [ ]:
from sklearn.linear_model import Ridge

# Load the features and ratings
features_df = pd.read_csv('features.tsv', sep='\t')
ratings_df = pd.read_csv('ratings.tsv', sep='\t')

# Extract features (exclude recipe_slug column, use all other columns including bias and tags)
X = features_df.drop('recipe_slug', axis=1).values

# Extract ratings
y = ratings_df['rating'].values

# Build ridge regression model with alpha=1 and fit_intercept=False
model = Ridge(alpha=1.0, fit_intercept=False)
model.fit(X, y)

print("Ridge Regression Model trained successfully!")
print(f"Model coefficients shape: {model.coef_.shape}")
print(f"First 10 coefficients (bias + first 9 tags): {model.coef_[:10]}")
print(f"Model intercept: {model.intercept_}")
print(f"Training R² score: {model.score(X, y):.4f}")

Ridge Regression Model trained successfully!
Model coefficients shape: (297,)
First 10 coefficients (bias + first 9 tags): [ 0.34190801  0.01993957  0.16884875 -0.07561846 -0.06602407  0.09272913
  0.15423144 -0.04786568  0.06863705  0.06013066]
Model intercept: 0.0
Training R² score: 0.9540


Save the coefficients of this model in a file "model.tsv" with columns "recipe_tag" and "coefficient".
Do not add anything for the `intercept_` attribute of a scikit-learn model; this will be covered by the coefficient for the bias column added in part 2.

In [ ]:
# Get the feature column names (excluding recipe_slug)
feature_columns = features_df.drop('recipe_slug', axis=1).columns.tolist()

# Create a dataframe with recipe_tag and coefficient columns
model_df = pd.DataFrame({
    'recipe_tag': feature_columns,
    'coefficient': model.coef_
})

# Save to model.tsv
model_df.to_csv('model.tsv', sep='\t', index=False)

print("Model coefficients saved successfully!")
print(f"Shape: {model_df.shape}")
print(f"\nFirst 10 rows:")
print(model_df.head(10))

Model coefficients saved successfully!
Shape: (297, 2)

First 10 rows:
     recipe_tag  coefficient
0          bias     0.341908
1       alfredo     0.019940
2        almond     0.168849
3      american    -0.075618
4     appetizer    -0.066024
5    appetizers     0.092729
6         apple     0.154231
7  asiancuisine    -0.047866
8     asparagus     0.068637
9       avocado     0.060131


Submit "model.tsv" in Gradescope.

## Part 4: Recipe Estimates

Use the recipe model to estimate the score of every recipe.
Save these estimates to a file "estimates.tsv" with columns recipe_slug and score_estimate.

In [ ]:
# YOUR CHANGES HERE



Submit "estimates.tsv" in Gradescope.

## Part 5: LinUCB Bounds

Calculate the upper bounds of LinUCB using data corresponding to trying every recipe once and receiving the rating in "ratings.tsv" as the reward.
Keep the ridge regression regularization parameter at 1, and set LinUCB's $\alpha$ parameter to 2.
Save these upper bounds to a file "bounds.tsv" with columns recipe_slug and score_bound.

In [ ]:
# YOUR CHANGES HERE

...

Submit "bounds.tsv" in Gradescope.

## Part 6: Make Online Recommendations

Implement LinUCB to make 100 recommendations starting with no data and using the same parameters as in part 5.
One recommendation should be made at a time and you can break ties arbitrarily.
After each recommendation, use the rating from part 1 as the reward to update the LinUCB data.
Record the recommendations made in a file "recommendations.tsv" with columns "recipe_slug", "score_bound", and "reward".
The rows in this file should be in the same order as the recommendations were made.

Hint: do not remove recipes after each recommendation.
Repeating recommendations is expected.

In [ ]:
# YOUR CHANGES HERE

...

Submit "recommendations.tsv" in Gradescope.

## Part 7: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgements are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.


Submit "project.ipynb" in Gradescope.